# Machine learning #

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
import itertools
import warnings
warnings.filterwarnings("ignore")

## ???? 0 - ???????? ???????? ##

In [ ]:
df = pd.read_csv('dataset_after_EDA.csv', engine='python')
df = df.dropna() # ? ???????? ???????? ???????? ????? ?????????? EDA ????????? ????????, ????????? ????? ?????????????
df.head()

????????, ????? ??????? ???????? ??? ??????? ????? ????????????? ? ???????? ????? ?????????

*   `title` - ???????? ?????
*   `publishedAt` - ????? ?????????? ?????, ??????? ???????????: "DAY TIME"
    *   DAY - ??????????? ???? ??????? ????? ? ??????? YYYY-MM-DD
    *   TIME - ????? ? ??????? HH:MM:SS


*   `channelTitle` - ???????? ??????, ?? ??????? ????? ???? ????????????
*   `category` - ????????? ?????

*   `trending_date` - ????, ? ??????? ????? ?????? ? ?????? ???????

*   `tags` - ????, ????????????? ??????? ? ?????

*   `view_count` - ?????????? ??????????
*   `likes` - ?????????? ??????
*   `comment_count` - ?????????? ???????????? ??? ?????
*   `comments_disabled` - ????????? ?? ??? ????? ???????????
*   `ratings_disabled` - ????????? ?? ??????????? ????????????? ?????????? ?????? ??? ?????
* `description` - ???????? ?????
* `country` - ??????, ? ??? ?????? ?????? ?????
* `subscriber_count` - ?????????? ??????????? ??????
* `daytime` - ????? ?????, ? ??????? ???? ???????????? ?????
    *   0 - ????? ???????????? ????? (? 18:00 ?? 6:00)
    *   1 - ????? ???????????? ???? (? 6:00 ?? 18:00)
* `times_appeared` - ?????????? ?????, ? ??????? ????? ?????? ? ??????

## ???? 1 - ?????????????? ?????????? ?????? ??? ????? ##



#### ?????? 1 - ??????? ???????? ????????? ####

? ??????? ??????? ?????? ???????? ????????? ????????? ??????????? ?????????? ?????? ??? ?????. ??????????????, ? ???????? ??????? ?????????? ?????????? ??????? `likes`, ? ???????? ??????????????? ????????? - `category`, `view_count`, `comment_count`, `country`, `subscriber_count`, `times_appeared`.

??????? ??????? ?? ????????????? ? ???????? ?????? ? ????????? 80 ?? 20.

????? ???????? ????? ?????? ?? ???????????? ????????????? ?????? ? ???????.

In [ ]:
df = df.drop_duplicates(subset = 'title')
df_encoded = pd.get_dummies(df, columns=['category', 'country'])

df_train, df_test = train_test_split(df_encoded, test_size = 0.2, random_state = 42)

X_train = df_train.drop(['title', 'publishedAt', 'channelTitle', 'trending_date', 'tags', 'likes', 'comments_disabled', 'ratings_disabled', 'description', 'daytime'], axis = 1)
X_test = df_test.drop(['title', 'publishedAt', 'channelTitle', 'trending_date', 'tags', 'likes', 'comments_disabled', 'ratings_disabled', 'description', 'daytime'], axis = 1)


y_train = df_train['likes']
y_test = df_test['likes']

????????, ??? ????? ????? ??????????????? ?????????? ???? ??? ?????????????? ???????? - `category` ? `country`. ?? ????????? ????? One Hot Encoding ?? ?????????? pandas ????? ????????????? ??????? get_dummies

????, ??????? ? ?????? ???? ??????, ??????? ???????????? ??????? ??????

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

In [ ]:
plt.figure(figsize = (8, 6))

fig, ax = plt.subplots(figsize=(12, 8))
ax.grid(which='both')

plt.hist(pred, bins = 40, density = True, label = '???????????')
plt.title('??????????? pred', fontsize = 15)
plt.xlabel('????????????? ???????? (???)', fontsize = 10)
plt.ylabel("???????", fontsize = 10)
sns.distplot(pred, hist = False, label = '????????? ?????????????')
plt.legend(loc = 'upper right');

????????????? ???????????? ????? ?????? ?????? ?? ????????? ????? ?????????? ?????????????. ????? ?? ??????? ??????? ???????

????? ??? ?????? ????????? ?????? ????????? ???????? ????????? ??????

1. `Mean Squared Error` - ?????????????????? ??????

In [ ]:
mean_squared_error(y_test, pred)

???????? ?????????? ??????? ???????? ?????????????????? ??????. ??? ???? ???????? MSE, ??? ????? ?????? ???????? ????????????? ??????? ??????????. ? ?????????? ?????????? ?????????????? ???? ??????, ????? ??????? ???????? ???????

2. `Mean Absolute Error` - ??????? ?????????? ??????

In [ ]:
mean_absolute_error(y_test, pred)

?????-???? ???????? ??????? ???????? ???????. MAE ???????? ??????? ?????????? ??????? ????? ?????????????? ?????????? ?????? ? ???????????? ?????????? ??????? ??????????. ? ????? ??????, ????????? ?????? ????????? ? ??????? ?? 32494 ????? ??? ??????????????. ???????? ??????? ???????? ?????????? ????? ?????? ?? ??????????? ??????????? ?????????? `likes`

In [ ]:
mean_absolute_error(y_test, pred) / df['likes'].std()

?????????? ????????????? ????? ??????? ???????? ?? ???????? ?????????? ????? 25% ?? ???????????? ?????????? - ??????????, ?? ??? ??? ??????? ??????? ??????

3. `R2` - ??????????? ????????????

In [ ]:
r2_score(y_test, pred)

???????? ??????? ?????????? ??????? ??????? ? ???, ??? ?????? ????????? ????? 74% ???????????? ??????? ??????????. ? ?????? ??????, ????? 74% ???????? ? ??????? ?????????? ????? ???? ????????? ??? ??????????? ? ?????????????? ????????? ???? ??????. ?????????? 26% ???????????? ????? ???? ??????? ? ??????? ?????????, ??????? ?????? ?? ?????????.

????, ?????? ??????? ?????? ???????? ?????????, ???????? ?? ????? ??????? ???????? ??????, ???????? ?? ???????????? ??????? ?????????? ???????????? ????????????. ? ????????? ?????? ????????? ?????????? ????????? ?????????????? ????????? ??????

#### ?????? 2 - ?????????? ??????? ???????? ????????? ####

??? ?????? ??????????? ???? ???????? ?????????? ? ??????? ?? 0 ?? 1

In [ ]:
scaler = preprocessing.MinMaxScaler()

numeric_data = X_train.select_dtypes([np.number])
numeric_features = numeric_data.columns

X_train_scaled = scaler.fit_transform(X_train[numeric_features])
X_test_scaled = scaler.transform(X_test[numeric_features])

????? ???????? ??????? ?? ????? ??? ?????? ???????? ??????????? ??????????? ?????????? ????? ???????? ?????????, ?????????? ??????? ??????????????? ?? ????? - GridSearchCV. ? ???????? ??????? ?????? ?????????????????? ????? ????????????? ???????? MSE

In [ ]:
params = {'fit_intercept': [True, False],
          'copy_X': [True, False],
          'n_jobs': [-1, 0, 1, 2, 3, 4]}

searcher = GridSearchCV(LinearRegression(), param_grid = params, scoring = 'neg_mean_absolute_error', cv = 4)

searcher.fit(X_train_scaled, y_train)

print(f'??????????? ????? ??????????? - {searcher.best_params_}, ???????? Negative MSE - {searcher.best_score_}')

In [ ]:
model = LinearRegression(copy_X = True, fit_intercept = True, n_jobs = -1)
model.fit(X_train_scaled, y_train)
pred = model.predict(X_test_scaled)

????? ??? ?????? ???????????????? ?????? ????????? ???????? ??? ??? ?? ??????

1. `Mean Squared Error` - ?????????????????? ??????

In [ ]:
mean_squared_error(y_test, pred)

??? ??? ???????? ?????????? ??????? ???????? ?????????????????? ??????. ?????????? ?????????? ???? ?? 0.01%

2. `Mean Absolute Error` - ??????? ?????????? ??????

In [ ]:
mean_absolute_error(y_test, pred)

?????-???? ???????? ??????? ???????? ???????. ????????? ????????? ?? 0.03%

3. `R2` - ??????????? ????????????

In [ ]:
r2_score(y_test, pred)

???????? ??????? ??????????? ????? ?? 0.004%

? ?????, ????????????? ???????? ??????????????? ?? ????? ?? ???? ???????? ???????????. ?? ????????? ??? ???, ??? ???????? ????????? ???????? ???????????? ??????? ???????, ? ?? ?????????????????? ????? ???? ?? ?????? ???????? ?? ?????? ???????????????.

## ???? 2 - ?????????????? ?????????? ???????????? ??? ????? ##



#### ??? 1 - ??????? ?????? RandomForestClassifier ####

??? ????, ????? ??????????? ???????????? ???????? ??????? ??????????, ?????? ?????? ?????????? ???? (RandomForestClassifier) ?? ?????????? sklearn.

??? ????? ??? ????? ??????????? ????????? ????? One Hot Encoding ??? ?????????? `category`. ? ???? ??? ?? ????????? ?????? ?? ???????????? ??????? `country`, ??? ??? ??????? ? ?????????? ??????????? ????? ??????????? ???????????? ? ???????, ? ?????? ??????? ??????? ?????? ?????.

????, ? ???????? ??????? ?????????? ?????????? ??????? `comments_disabled`, ? ???????? ??????????????? ????????? - `category`, `view_count`, `likes`, `subscriber_count`, `ratings_disabled`.

????? ??????????? ??????? `rating_disabled` ? `comments_disabled` - ?????? ??????? ?????????? True ? False ????? ???????? ? ????????? ?????????? 1 ? 0

In [ ]:
df_encoded = pd.get_dummies(df, columns=['category'])

df_encoded['ratings_disabled'] = df_encoded['ratings_disabled'].apply(lambda x: 1 if x == 'True' else 0)
df_encoded['comments_disabled'] = df_encoded['comments_disabled'].apply(lambda x: 1 if x == 'True' else 0)

df_train, df_test = train_test_split(df_encoded, test_size = 0.2, random_state = 42)

X_train = df_train.drop(['title', 'publishedAt', 'channelTitle', 'trending_date', 'tags', 'description', 'daytime', 'country', 'comment_count', 'comments_disabled', 'times_appeared'], axis = 1)
X_test = df_test.drop(['title', 'publishedAt', 'channelTitle', 'trending_date', 'tags', 'description', 'daytime', 'country', 'comment_count', 'comments_disabled', 'times_appeared'], axis = 1)


y_train = df_train['comments_disabled']
y_test = df_test['comments_disabled']

????? ?????????? ???? ??????, ?????????? ??????? ?? ???????? ? ????????????? ?? ?????? ?????????? ? ???????? ???????? ?????? ? ????????? ???????????? ??????? ??????

In [ ]:
model = RandomForestClassifier(n_estimators=1000,
                               bootstrap = True,
                               max_features = 'sqrt',
                               random_state = 777)

model.fit(X_train, y_train)

pred = model.predict(X_test)
pred_proba = model.predict_proba(X_test)[:, 1]

#### ?????? ?????? ####

??? ?????? ????????? ?????? ?????????? ???? ????????? ???????? ROC AUC

?????????? ROC AUC ???????? ??????? ??? ROC ??????, ??????? ?????????? ??????????? ????? ????? ????? ????????????? ? ????? ????? ????????????? ????????????? ??? ???????????? ?????? ?????????????.

In [ ]:
roc_value = roc_auc_score(y_test, pred_proba)
roc_value

In [ ]:
base_fpr, base_tpr, _ = roc_curve(y_test, [1 for _ in range(len(y_test))])
model_fpr, model_tpr, _ = roc_curve(y_test, pred_proba)

plt.figure(figsize = (8, 6))

fig, ax = plt.subplots(figsize=(12, 8))
ax.grid(which='both')

plt.plot(model_fpr, model_tpr, color = 'red', label = '????????? ??????')
plt.plot(base_fpr, base_tpr, color = 'blue', label = '????????? ?????????????')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc = 'upper left');

???????? ???????? ??????? ??? ROC ?????? - 0.69, ???? ?????? ???? ?????????? ????? ?????????? ??????????????!

????? ?????? ???? ?????? ? ??????? ?????????? ??????? ????????

In [ ]:
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Oranges):
    """
    ??????? ??? ?????????? ??????? ???? ????? ?? ?????????: http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    plt.figure(figsize = (7, 7))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title, size = 14)
    plt.colorbar(aspect=4)
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45, size = 10)
    plt.yticks(tick_marks, classes, size = 10)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.

    # Labeling the plot
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt), fontsize = 10,
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.grid(None)
    plt.tight_layout()
    plt.ylabel('??????????', size = 12)
    plt.xlabel('???????????? ??????', size = 12);

In [ ]:
cm = confusion_matrix(y_test, pred)
plot_confusion_matrix(cm, classes = ['?? ?????????', '?????????'],
                      title = '??????? ???????? ??? ?????????? ????????????', normalize=False)

? ?????????? ???????? ????????? ????????:
> `True Positive` ? `True Negative` - ???????? ???? ???? ????? ??????????? 531 ?????????? ????????????

> `False Negative` ? `False Positive` - ???????? ???? ???? ???????? 9 ?????? ??? ?????????????? ?????????? ????????????

????? ??????? ?????????? ????????? ?????? ?? ????????? ????? ??????? ??????????? ???????????? ? ?????????????? ?????????????



?????? ????????? ?? ??????? ??? ??? ???? ?????????? ?? ?????????????? ????????? ???? ??????

In [ ]:
impact = pd.DataFrame({'feature': list(X_train),
                   'importance': model.feature_importances_}).\
                    sort_values('importance', ascending = False)
impact.head()

????????? ???????? ?? ???????????:

> `likes`, `subscriber_count`, `view_count` - ?????, ????????? ??????? ?????????? ?????? ??? ??????????, ?????? ?????, ????????? ?????-?? ??????????, ???????? ?????????????? ????. ?????? ??????? ?? ????????? ????????????? ???????????? ??? ?????, ?????????? ??? ?????? ????? ??????????

> ??????????? ????????? `News & Politics` - ?????, ????????? ? ????????? ? ?????????, ????? ?????????? ??????? ?????????? ????????????, ??????? ??????, ?????????? ? ????????? ????? ??????. ??? ????? ???????? ? ??????????????? ????????????? ??????, ????????????? ??? ???????? ????????.

> ??????????? ????????? `Entertainment` - ????? ???????????????? ????????? ????? ?????????? ??????? ?????????, ? ??????????? ????? ????????? ?????????? ??? ?????????????? ????????????. ?????????? ???????????? ???????? ????????????? ?????????? ????????? ? ??????? ????? ???????? ???????????? ??? ????????.







#### ??? 2 - ????? ???????? ??????????? ?????????? ????? RandomizedSearchCV ####

????? ??????????? ??????????? ?????????? ? ?? ???????? ????????? ??????? ???????? ? ??????? ????? ?????? ???????? ?????????? ??????????? ???????????

In [ ]:
param_grid = {
    'n_estimators': np.linspace(10, 200).astype(int),
    'max_depth': [None] + list(np.linspace(3, 20).astype(int)),
    'max_features': ['auto', 'sqrt', None] + list(np.arange(0.5, 1, 0.01)),
    'max_leaf_nodes': [None] + list(np.linspace(10, 50, 1000).astype(int)),
    'min_samples_split': [2, 5, 10],
    'bootstrap': [True, False]
}

estimator = RandomForestClassifier(random_state = 777)

rs = RandomizedSearchCV(estimator, param_grid, n_jobs = -1,
                        scoring = 'roc_auc', cv = 3,
                        n_iter = 10, verbose = 1, random_state=777)

rs.fit(X_train, y_train)
rs.best_params_

In [ ]:
model = RandomForestClassifier(n_estimators = 192,
                               min_samples_split = 10,
                               max_leaf_nodes = 35,
                               max_features = 0.6000000000000001,
                               max_depth = 12,
                               bootstrap = True,
                               random_state = 777)

model.fit(X_train, y_train)

pred = model.predict(X_test)
pred_proba = model.predict_proba(X_test)[:, 1]

In [ ]:
roc_value = roc_auc_score(y_test, pred_proba)
roc_value

In [ ]:
base_fpr, base_tpr, _ = roc_curve(y_test, [1 for _ in range(len(y_test))])
model_fpr, model_tpr, _ = roc_curve(y_test, pred_proba)

plt.figure(figsize = (8, 6))

fig, ax = plt.subplots(figsize=(12, 8))
ax.grid(which='both')

plt.plot(model_fpr, model_tpr, color = 'red', label = '????????? ??????')
plt.plot(base_fpr, base_tpr, color = 'blue', label = '????????? ?????????????')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc = 'upper left');

?? ?????????, ????? ?????? ???? ??? ????? ?????? ?????????? ROC AUC - 0.69 ?????? 0.67

## ???? 3 - ?????????? ##

????, ?? ??????? ??? ??????: ??????? ???????? ????????? (?????????????? ?????????? ?????? ??? ?????) ? ????????? ??? (?????????????? ?????????? ???????????? ??? ?????), ???????? ???????????? ??????? ??????, ?????????? ??????. ????? ???? ???? ??????????? ??????? ???????? ???????? ????????? ??????? - ? ????? ??????? ?? ????????? ??????? ??????????????? ?? ?????, ? ?????? ? ???????? ?????????? ???? ??????????? ??????? ???????? ?????????? ?????? ????? ???????????? ?????? ? ??????? ?? 0 ?? 1, ? ????? ????????? ????????? ??????????????? ???????????.

? ??????????, ????????? ???? ???????? ????????? ???? ???? ????? ??????? ???????? ??????????? ??????, ?????? ??? ??? ??????????? ??????? ???????? ?????????? ????????????? ???????? ?? ???????????, ?????? ????????????? ? ?? ????? ??????? ????????? ???????????? ???????.

????? ???????? ??????????????? ? ?????? ?????????? ???? ???????? ??????? ROC AUC ? ????? ???????????, ?????? ????????? ?????? ??? ??? ????????????? ????? ?????????? ??????????????. ???????? ???????? ?? ???????? ????????? ???? ?????????? ??????????? ????? ?????? ?????????? ???????????? ? ??????? ????????????, ?????? ??? `likes`, `subscriber_count`, `view_count` ? ??????????????? ??????????? ? ?????????? `News & Politics` ??? `Entertainment`. ???????????? ??, ??? ?????? ??????????? ?? ???? ??????? ?? ????? ??????????????? ??????? - ?????? ????? ? ???, ??? ????? ???????? ? ??????? ?????????? ????? ???? ????????? ??? ??????????? ? ?????????????? ????????? ???? ??????.